# Especialista *prestart* H60 — el candidato final

Este es el cuaderno del modelo que acaba siendo **el candidato congelado** del proyecto.
Después de comprobar que los baselines direccionales mueren con latencia y que el deep
learning aporta representación pero no política, volvemos a un modelo tabular, pero muy
enfocado.

La idea, que venía del análisis exploratorio: la **ventana anterior a la apertura del
mercado** (*prestart*) concentra actividad atípica del libro, así que entrenamos un
**especialista** solo para esa ventana en lugar de un modelo generalista.

Se prueban dos recortes de ventana y dos conjuntos de características:

- **universos**: `strict_0_60_all` (los 60 s previos) y `strict_45_60_early` (solo
  −60 a −45 s, el arranque del prestart);
- **feature sets**: `full_features` y `no_clock` (sin hora/minuto/segundo, para que el
  modelo no aprenda atajos triviales del reloj).

El modelo es un **HistGradientBoosting de dos cabezas**: un *regresor* que predice el
valor esperado de la operación (`exec_net_cost_H60`) y un *clasificador* que predice si
el relleno será sano (`healthy_fill_proxy_H60`). La selección de la política se hace
**solo con validación**.

> El entrenamiento vive en `scripts/experiments/complex_v1a_prestart_h60_specialist_v1.py`.
> Aquí cargamos y leemos los resultados.


In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path('../data/experiments/complex_v1a_prestart_h60_specialist_v1')
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
universe = pd.read_csv(OUT / 'specialist_universe_summary.csv')
metrics = pd.read_csv(OUT / 'specialist_model_metrics.csv')
selected = pd.read_csv(OUT / 'specialist_selected_policies.csv')
buckets = pd.read_csv(OUT / 'specialist_selected_bucket_breakdown.csv')
days = pd.read_csv(OUT / 'specialist_selected_day_breakdown.csv')
decision


## 1. Universo de operaciones

Antes de mirar modelos, conviene saber **cuántas operaciones candidatas** deja cada
combinación de ventana y filtros estrictos. Si un universo se queda con muy pocas
operaciones, cualquier resultado posterior será frágil por falta de soporte. Esta tabla
es el "censo" del que partimos.


In [ ]:
universe.round(5)


## 2. Métricas de los modelos

Para cada universo y feature set, las métricas de las dos cabezas: el ajuste del
**regresor de EV** (error y capacidad de ordenar) y el del **clasificador de salud**
(AUC). Recuérdese que el criterio último no es el ajuste estadístico, sino lo que
ocurre con la política económica; estas métricas son el primer filtro.


In [ ]:
metrics.round(5)


## 3. Políticas seleccionadas

Las políticas que sobreviven a la selección **hecha solo sobre validación**: la
combinación de umbrales (EV predicho y probabilidad de salud) que maximiza el neto por
operación con soporte suficiente. Esta es la frontera entre "el modelo ve algo" y "el
modelo decide algo". El candidato que pasa de aquí es el que más tarde se **congela**
antes de ver los datos fuera de muestra.


In [ ]:
selected.round(5)


## 4. Desglose por buckets

Repartimos las operaciones en *buckets* para ver **de dónde sale el resultado**: no es
lo mismo un neto positivo repartido de forma homogénea que uno concentrado en un rincón
del espacio. Sirve para detectar si la ventaja es estructural o un artefacto de un
subgrupo concreto.


In [ ]:
buckets.round(5)


## 5. Desglose diario

La prueba de estabilidad: el resultado **día a día**. Una política defendible no puede
depender de un único día bueno. Aquí buscamos consistencia, y es justo lo que más adelante
se confirma en la validación fuera de muestra (5/5 días positivos sobre datos frescos).


In [ ]:
days.sort_values(['universe','feature_set','terminal_split','session_day']).round(5)


## Lectura y siguiente paso

De esta exploración sale el candidato que se **congela**: especialista de la ventana
`strict_45_60_early`, feature set `no_clock`, HGB de dos cabezas, con la política de
umbrales seleccionada solo sobre validación (EV predicho > 0,75 y probabilidad de salud
≥ 0,50).

A partir de aquí no se vuelve a tocar para "mejorar" los números: se le añade el *vol
gate* (por el cambio de régimen) y se valida fuera de muestra sobre datos capturados
después. Esa validación fresh es la que da +1,069 ticks netos por operación con 5/5 días
positivos. Ver `docs/14_especialista_prestart_h60.md` y `docs/16_validacion_fresh.md`.
